# Latent Space Alignment via Relative Representations

This notebook implements the core mechanics of the **Relative Representations** paradigm "Relative representations enable zero-shot latent space communication" https://arxiv.org/abs/2209.15430 I came accross while reading "The Platonic Representation Hypothesis" https://arxiv.org/abs/2405.07987.

### Core Theory
Different language models (e.g., **Qwen-2.5-0.5B** and **TinyLlama-1.1B**) reside in entirely disparate latent vector spaces due to differences in dimensionalities, architectures, and training. However, the *geometric relationships between shared concepts* (e.g., how close "market" is to "money" or "business") are highly invariant across models and modalities ( different languages).

By measuring the cosine similarities of target words against a shared set of **parallel anchor words**, we project absolute embeddings into a **coordinate-free relative space**. This allows zero-shot cross-lingual and cross-model alignment without requiring additional learnable parameters or linking layers.


## 1. Environment & Dependencies
> 🤖 **AI-Assisted**

We load libraries and select high-performance Apple Silicon GPU (`mps`) hardware if available.


In [1]:
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch.nn.functional as F
from scipy.stats import entropy

# Direct Apple Silicon / CUDA acceleration setup
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Target hardware accelerator: {device}")


/Users/abdullah/Pytorch/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/abdullah/Pytorch/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Target hardware accelerator: mps


## 2. Parallel Anchors and Test Sets
> 🤖 **AI-Assisted**

We define parallel vocabularies across English and Chinese. 


In [10]:
# Parallel Anchor sets used to construct the relative coordinate system
english_anchors = [
    "time", "year", "people", "way", "day", "man", "thing", "woman", "life", "child",
    "world", "school", "state", "family", "student", "group", "country", "problem", "hand", "part",
    "place", "case", "week", "company", "system", "program", "question", "work", "government", "number",
    "night", "point", "home", "water", "room", "mother", "area", "money", "story", "fact",
    "month", "lot", "right", "study", "book", "eye", "job", "word", "business", "issue",
    "side", "kind", "head", "house", "service", "friend", "father", "power", "hour", "game",
    "line", "end", "member", "law", "car", "city", "community", "name", "idea", "body",
    "information", "back", "parent", "face", "others", "level", "office", "door", "health", "person",
    "art", "war", "history", "party", "result", "change", "morning", "reason", "research", "girl",
    "guy", "moment", "air", "teacher", "force", "education", "foot", "boy", "age", "policy"
]

chinese_anchors = [
    "时间", "年", "人", "方式", "天", "男人", "事情", "女人", "生命", "孩子",
    "世界", "学校", "国家", "家庭", "学生", "团队", "国家", "问题", "手", "部分",
    "地方", "案件", "星期", "公司", "系统", "项目", "问题", "工作", "政府", "数字",
    "夜晚", "观点", "家", "水", "房间", "母亲", "区域", "金钱", "故事", "事实",
    "月份", "许多", "权利", "研究", "书", "眼睛", "工作", "词语", "商业", "议题",
    "面", "种类", "头", "房子", "服务", "朋友", "父亲", "力量", "小时", "游戏",
    "线", "结束", "成员", "法律", "汽车", "城市", "社区", "名字", "想法", "身体",
    "信息", "背部", "父母", "脸", "其他人", "水平", "办公室", "门", "健康", "人",
    "艺术", "战争", "历史", "党派", "结果", "改变", "早晨", "原因", "研究", "女孩",
    "家伙", "时刻", "空气", "老师", "力量", "教育", "脚", "男孩", "年龄", "政策"
]

# Target test words for evaluation
english_test = ["market", "guide", "source", "future", "nature", "action", "growth", "choice", "signal", "theory"]
chinese_test = ["市场", "指南", "来源", "未来", "自然", "行动", "增长", "选择", "信号", "理论"]


## 3. Embedding Extraction
> 🤖 **AI-Assisted**

Instead of writing duplicate loops for both models, we write a clean, generic function. 
To optimize, we directly forward inputs through the base transformer (`model.model` or `model.transformer`), skipping the huge token classifier head (`lm_head`).


In [9]:
def extract_embeddings(words, model, tokenizer, device):
    """Extracts base-layer mean-pooled hidden representations cleanly."""
    embeddings = []
    # Determine the model's base transformer dynamically
    base_transformer = getattr(model, "model", getattr(model, "transformer", model))
    
    with torch.no_grad():
        for word in words:
            inputs = tokenizer(word, return_tensors="pt").to(device)
            outputs = base_transformer(**inputs)
            
            # Mean-pool sequence length dimension to collapse tokens
            embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding.cpu())
            
    return torch.stack(embeddings)


## 4. Extracting Absolute Representations
> 🤖 **AI-Assisted**

We load both models and run our optimized extraction function to extract absolute embeddings for anchors and test words.


In [4]:
# --- Qwen 2.5 (0.5B Multilingual Model) ---
print("Loading Qwen 2.5 (0.5B)...")
qwen_model_name = "Qwen/Qwen2.5-0.5B-Instruct"
qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(qwen_model_name).to(device)
qwen_model.eval()

print("Extracting Qwen embeddings...")
qwen_anchors = extract_embeddings(chinese_anchors, qwen_model, qwen_tokenizer, device)
qwen_test = extract_embeddings(chinese_test, qwen_model, qwen_tokenizer, device)

# --- TinyLlama (1.1B English-only Model) ---
print("\nLoading TinyLlama (1.1B)...")
tiny_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tiny_tokenizer = AutoTokenizer.from_pretrained(tiny_model_name)
tiny_model = AutoModelForCausalLM.from_pretrained(tiny_model_name).to(device)
tiny_model.eval()

print("Extracting TinyLlama embeddings...")
tiny_anchors = extract_embeddings(english_anchors, tiny_model, tiny_tokenizer, device)
tiny_test = extract_embeddings(english_test, tiny_model, tiny_tokenizer, device)

print("\nAbsolute embedding sizes loaded:")
print(f"Qwen Anchors: {qwen_anchors.shape} | TinyLlama Anchors: {tiny_anchors.shape}")


Loading Qwen 2.5 (0.5B)...
Extracting Qwen embeddings...

Loading TinyLlama (1.1B)...
Extracting TinyLlama embeddings...

Absolute embedding sizes loaded:
Qwen Anchors: torch.Size([100, 896]) | TinyLlama Anchors: torch.Size([100, 2048])


## 5. Constructing Relative Coordinate Spaces
> ✍️ **My Work** — I designed the relative space projection and temperature-scaling approach.

We project absolute embeddings into the relative spaces using two modes:
1. **Raw Vector Space** (cosine similarities directly).
2. **Probability Space** (Softmax normalized with Temperature). Softmax with a low temperature (like $T=0.1$) acts as an amplifier, preserving the structural differences and mapping similarities cleanly into $[0.0, 1.0]$ for KL Divergence.


In [11]:
def build_relative_space(test_embeddings, anchor_embeddings, temperature=None):
    """Projects absolute embeddings to coordinate-free relative vectors."""
    # Standardize/Normalize absolute vectors
    test_norm = test_embeddings / test_embeddings.norm(dim=1, keepdim=True)
    anchors_norm = anchor_embeddings / anchor_embeddings.norm(dim=1, keepdim=True)
    
    # Direct pairwise cosine similarity matrix
    raw_similarity_space = test_norm @ anchors_norm.t()
    
    if temperature is not None:
        # Softmax with temperature keeps variance crisp for probability metrics (like KL)
        return torch.softmax(raw_similarity_space / temperature, dim=1).numpy()
    
    return raw_similarity_space.numpy()

# 1. Cosine similarity relative representation vectors
similarity_spaces_ch_cos = build_relative_space(qwen_test, qwen_anchors)
similarity_spaces_en_cos = build_relative_space(tiny_test, tiny_anchors)

# 2. Softmax-temperature scaling for probability distribution representations
similarity_spaces_ch_prob = build_relative_space(qwen_test, qwen_anchors, temperature=0.05)
similarity_spaces_en_prob = build_relative_space(tiny_test, tiny_anchors, temperature=0.05)


## 6. Ranking Metrics
> ✍️ **My Work** — Wrote the ranking evaluation logic.

To evaluate the alignment quality, we compute the rank of the correct translation. A rank of `0` is a perfect match.


In [12]:
def element_rank(arr, position):
    """Returns the sorted rank of the target position (0 = closest/best)."""
    arr = np.asarray(arr)
    ranks = np.argsort(np.argsort(arr))
    return ranks[position]


## 7. Metric 1: KL Divergence (Probability Profile Space)
> ✍️ **My Work** — KL divergence over softmax profiles was my own extension, not from the paper.

We measure similarity using the Information Theory metric **Kullback-Leibler (KL) Divergence** over the temperature-softmaced spaces. This is not from the paper but was made out of curousity 


In [13]:
ranks_kl = []
for j in range(len(english_test)):
    curr_kl_list = []
    for i in range(len(chinese_test)):
        # Compute KL over softmax-normalized probability profiles
        kl = entropy(similarity_spaces_en_prob[j], similarity_spaces_ch_prob[i])
        curr_kl_list.append(kl)
        
    rank = element_rank(curr_kl_list, j)
    ranks_kl.append(rank)
    print(f"Word: {english_test[j]} -> {chinese_test[j]} | Rank: {rank} (out of {len(english_test)-1})")

print(f"\nKL Divergence Average Rank: {np.mean(ranks_kl):.2f}")


Word: market -> 市场 | Rank: 1 (out of 9)
Word: guide -> 指南 | Rank: 3 (out of 9)
Word: source -> 来源 | Rank: 4 (out of 9)
Word: future -> 未来 | Rank: 0 (out of 9)
Word: nature -> 自然 | Rank: 0 (out of 9)
Word: action -> 行动 | Rank: 7 (out of 9)
Word: growth -> 增长 | Rank: 4 (out of 9)
Word: choice -> 选择 | Rank: 3 (out of 9)
Word: signal -> 信号 | Rank: 0 (out of 9)
Word: theory -> 理论 | Rank: 0 (out of 9)

KL Divergence Average Rank: 2.20


## 8. Metric 2: Cosine Similarity (Vector Space)
> ✍️ **My Work** — Designed the cosine-distance ranking evaluation.

Instead of probability distributions, we treat the relative projections as standard vectors in $\mathbb{R}^k$ and compare them directly using standard vector Cosine Similarity.


In [8]:
ranks_cos = []
for j in range(len(english_test)):
    curr_cos_list = []
    for i in range(len(chinese_test)):
        # Calculate cosine similarity directly between the raw relative representation vectors
        sim_val = F.cosine_similarity(
            torch.tensor(similarity_spaces_en_cos[j]),
            torch.tensor(similarity_spaces_ch_cos[i]),
            dim=0
        ).item()
        
        # Convert to distance metric (1 - sim) so smaller is closer
        curr_cos_list.append(1.0 - sim_val)
        
    rank = element_rank(curr_cos_list, j)
    ranks_cos.append(rank)
    print(f"Word: {english_test[j]} -> {chinese_test[j]} | Rank: {rank} (out of {len(english_test)-1})")

print(f"\nRelative Vector Cosine Similarity Average Rank: {np.mean(ranks_cos):.2f}")


Word: market -> 市场 | Rank: 1 (out of 9)
Word: guide -> 指南 | Rank: 8 (out of 9)
Word: source -> 来源 | Rank: 5 (out of 9)
Word: future -> 未来 | Rank: 2 (out of 9)
Word: nature -> 自然 | Rank: 0 (out of 9)
Word: action -> 行动 | Rank: 6 (out of 9)
Word: growth -> 增长 | Rank: 4 (out of 9)
Word: choice -> 选择 | Rank: 4 (out of 9)
Word: signal -> 信号 | Rank: 5 (out of 9)
Word: theory -> 理论 | Rank: 0 (out of 9)

Relative Vector Cosine Similarity Average Rank: 3.50
